# ⚡ Text2SQL QLoRA Fine-Tuning Notebook
**Fine-Tuning Qwen-2.5-Coder-7B-Instruct using Unsloth & HuggingFace PEFT on a Free Google Colab T4 GPU**

This notebook trains a 4-bit quantized LoRA adapter on your custom DuckDB Text-to-SQL dataset.

In [ ]:
# 1. Install Unsloth, HuggingFace Transformers, PEFT, and BitsAndBytes
!pip install --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers trl peft accelerate bitsandbytes triton

In [ ]:
# 2. Load Base Model (Qwen-2.5-Coder-7B) in 4-bit Precision
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
load_in_4bit = True  # 4-bit NF4 Quantization

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "Qwen/Qwen2.5-Coder-7B-Instruct",
    max_seq_length = max_seq_length,
    load_in_4bit = load_in_4bit,
)

In [ ]:
# 3. Configure QLoRA Adapters (r=16, lora_alpha=32)
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 32,
    lora_dropout = 0, # Optimized 0 for Unsloth
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

In [ ]:
# 4. Load Training Dataset (finetune_dataset.jsonl)
from datasets import load_dataset

dataset = load_dataset("json", data_files="finetune_dataset.jsonl", split="train")

def format_prompts(examples):
    texts = []
    for messages in examples["messages"]:
        formatted_text = f"<|im_start|>system\n{messages[0]['content']}<|im_end|>\n<|im_start|>user\n{messages[1]['content']}<|im_end|>\n<|im_start|>assistant\n{messages[2]['content']}<|im_end|>"
        texts.append(formatted_text)
    return { "text" : texts }

dataset = dataset.map(format_prompts, batched = True)

In [ ]:
# 5. Execute Supervised Fine-Tuning (SFTTrainer)
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60, # 60 iterations for fast Colab training
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

trainer_stats = trainer.train()

In [ ]:
# 6. Save Fine-Tuned LoRA Adapter Weights & Push to HuggingFace Hub
model.save_pretrained("lora_model") # Local save
tokenizer.save_pretrained("lora_model")

# Optional: Push directly to Hugging Face Hub (Uncomment and replace username)
# model.push_to_hub_merged("your-hf-username/text2sql-qwen2.5-duckdb", tokenizer, save_method = "merged_16bit")